# Stage 2: Feature Engineering (Trích xuất Đặc trưng HRV)
Nhiệm vụ: Dùng kỹ thuật cửa sổ trượt (Sliding Window 5 phút) quét qua file dữ liệu sạch, tính toán các chỉ số y khoa như SDNN, RMSSD, LF/HF và lưu lại thành Dataframe cuối cùng.

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks, welch
from scipy import interpolate
import os

# 1. Đọc dữ liệu đã được làm sạch từ Bước 1
filepath = '../data/processed/clean_PHU_30_PHUT.csv'
df = pd.read_csv(filepath)
print(f"Đã đọc {len(df)} dòng dữ liệu từ {filepath}")

In [ ]:
# 2. ĐỊNH NGHĨA CÁC HÀM TOÁN HỌC TRÍCH XUẤT ĐẶC TRƯNG Y KHOA

def calculate_time_domain(rr_intervals):
    """Tính các đặc trưng miền thời gian (Time-domain)"""
    if len(rr_intervals) < 2:
        return np.nan, np.nan, np.nan
    
    mean_nn = np.mean(rr_intervals)          # Nhịp tim trung bình
    sdnn = np.std(rr_intervals)              # Độ lệch chuẩn (Biến thiên tổng thể)
    diff_nn = np.diff(rr_intervals)
    rmssd = np.sqrt(np.mean(diff_nn**2))     # RMSSD (Căng thẳng/Thư giãn)
    
    return mean_nn, sdnn, rmssd

def calculate_frequency_domain(rr_intervals):
    """Tính các đặc trưng miền tần số (Frequency-domain) bằng thuật toán Welch"""
    if len(rr_intervals) < 10:
        return np.nan, np.nan, np.nan
        
    # Interpolation (Biến khoảng cách rời rạc thành đồ thị liên tục để chạy Fourier)
    t = np.cumsum(rr_intervals) / 1000.0 # Đổi ra giây
    t -= t[0]
    
    fs_interp = 4.0 # Lấy mẫu giả định 4Hz cho HRV
    t_interp = np.arange(0, t[-1], 1/fs_interp)
    f_interp = interpolate.interp1d(t, rr_intervals, kind='cubic', fill_value="extrapolate")
    rr_interp = f_interp(t_interp)
    
    # Chạy Welch Periodogram
    f, pxx = welch(rr_interp, fs=fs_interp, nperseg=min(len(rr_interp), 256))
    
    # Tìm dải LF (0.04 - 0.15 Hz) và HF (0.15 - 0.40 Hz)
    # Dùng np.trapezoid thay cho np.trapz (vì Numpy phiên bản mới đã loại bỏ trapz)
    lf = np.trapezoid(pxx[(f >= 0.04) & (f < 0.15)], f[(f >= 0.04) & (f < 0.15)])
    hf = np.trapezoid(pxx[(f >= 0.15) & (f < 0.40)], f[(f >= 0.15) & (f < 0.40)])
    
    lf_hf_ratio = lf / hf if hf > 0 else np.nan
    return lf, hf, lf_hf_ratio

In [ ]:
# 3. CHẠY THUẬT TOÁN CỬA SỔ TRƯỢT (SLIDING WINDOW)
fs = 100.0 # Tần số lấy mẫu 100Hz
window_size_ms = 5 * 60 * 1000 # Cửa sổ 5 phút = 300.000 ms
step_size_ms = 1 * 60 * 1000   # Trượt mỗi 1 phút = 60.000 ms

features_list = []

# Tính tổng thời gian của cả file
start_time = df['Time(ms)'].iloc[0]
end_time = df['Time(ms)'].iloc[-1]

print(f"Bắt đầu quét dữ liệu bằng cửa sổ trượt...")
current_start = start_time

while current_start + window_size_ms <= end_time:
    current_end = current_start + window_size_ms
    
    # Cắt lấy 1 khúc dữ liệu 5 phút
    window_df = df[(df['Time(ms)'] >= current_start) & (df['Time(ms)'] < current_end)]
    
    # 1. Tìm đỉnh nhịp tim trong 5 phút này
    min_distance = int(0.3 * fs)
    peaks, _ = find_peaks(window_df['IR_Filtered'], distance=min_distance, prominence=10)
    
    # 2. Tính R-R Intervals (khoảng cách giữa các đỉnh, tính bằng milli-giây)
    peak_times = window_df['Time(ms)'].iloc[peaks].values
    rr_intervals = np.diff(peak_times)
    
    # Lọc bỏ các R-R sai số y khoa (nhỏ hơn 300ms hoặc lớn hơn 2000ms)
    rr_intervals = rr_intervals[(rr_intervals >= 300) & (rr_intervals <= 2000)]
    
    # 3. Trích xuất Đặc trưng nếu có đủ nhịp tim
    if len(rr_intervals) > 30:
        mean_nn, sdnn, rmssd = calculate_time_domain(rr_intervals)
        lf, hf, lf_hf_ratio = calculate_frequency_domain(rr_intervals)
        
        features_list.append({
            'Window_Start(ms)': current_start,
            'Mean_NN': mean_nn,
            'SDNN': sdnn,
            'RMSSD': rmssd,
            'LF': lf,
            'HF': hf,
            'LF_HF_Ratio': lf_hf_ratio,
            'Label': 'Sitting' # <--- GÁN NHÃN LÀ NGỒI IM Ở ĐÂY
        })
        
    # Trượt cửa sổ đi thêm 1 phút
    current_start += step_size_ms

# Gom kết quả thành Bảng (DataFrame)
features_df = pd.DataFrame(features_list)
print(f"\nHOÀN THÀNH! Trích xuất thành công {len(features_df)} mẫu dữ liệu (Data points) từ file 30 phút.")
features_df.head()

In [ ]:
# 4. LƯU TẬP DỮ LIỆU CUỐI CÙNG
os.makedirs('../data/features', exist_ok=True)
save_path = '../data/features/features_dataset.csv'
features_df.to_csv(save_path, index=False)
print(f"Đã xuất Bảng Đặc trưng ra file: {save_path} - Sẵn sàng để huấn luyện AI!")